### Generate Noisy Data

Estimates signal levels independently per frequency and direction and then adds noise with a given SNR.

In [31]:
import numpy as np
from scipy.io import loadmat, savemat

In [ ]:
# Load data

fpath = "..." # Path to "four_target_phantom.mat"
data_raw = loadmat(fpath)
data = data_raw["u_ft"]
data = np.moveaxis(data, -1, 0)

In [ ]:
# Generate noisy data

snr_vals = [None, 60, 40, 20, 0] # SNR in dB

data_noisy = []

for snr in snr_vals:
    if snr is not None:

        sig_level = np.mean(np.abs(data) ** 2, axis = (1,2,3), keepdims=True) # Calculate signal level across spatial dimensions, independent in MEG and frequency
        noise_sigma = np.sqrt(10 ** (-snr/10) * sig_level) # Calculate sigma
        noise_field_real = np.random.normal(0., noise_sigma / np.sqrt(2.), data.shape)
        noise_field_imag = np.random.normal(0., noise_sigma / np.sqrt(2.), data.shape)
        noise_field = noise_field_real + 1j * noise_field_imag
        dnoise = data + noise_field
        data_noisy.append(dnoise)
    
    if snr is None:
        data_noisy.append(data)

data_noisy = np.stack(data_noisy, axis = 0)


In [37]:
# Save noisy data

ddict = {"data": data_noisy}

np.save("bfem_noisy.npy", data_noisy)
savemat("bfem_noisy_mat.mat", ddict)